
# 🚀 Vision RAG (BLIP + Groq) - Production Ready

Features:
- BLIP image captioning (true vision)
- Groq LLM refinement (10 words)
- Follow-up Q&A (chat-style)
- Professional Gradio UI
- API key input


In [ ]:

!pip install -U groq gradio pillow transformers torch


In [ ]:

from groq import Groq
from PIL import Image
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch


In [ ]:

# Load BLIP model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


In [ ]:

def generate_caption_blip(image):
    inputs = processor(image, return_tensors="pt")
    out = model_blip.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)


In [ ]:

def refine_caption(caption, api_key):
    client = Groq(api_key=api_key)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"Summarize this in exactly 10 words:\n{caption}"
        }],
    )
    return response.choices[0].message.content


In [ ]:

def ask_question(caption, question, api_key):
    client = Groq(api_key=api_key)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"Context: {caption}\nQuestion: {question}"
        }],
    )
    return response.choices[0].message.content


In [ ]:

def pipeline(image, api_key):
    caption = generate_caption_blip(image)
    refined = refine_caption(caption, api_key)
    return caption, refined


In [ ]:

def app():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🧠 Vision RAG Assistant")

        with gr.Row():
            api_key = gr.Textbox(label="Groq API Key", type="password")
            image = gr.Image(type="pil")

        caption_out = gr.Textbox(label="BLIP Caption")
        refined_out = gr.Textbox(label="10-word Explanation")

        btn = gr.Button("Generate")
        btn.click(pipeline, inputs=[image, api_key], outputs=[caption_out, refined_out])

        gr.Markdown("## 💬 Ask Questions")

        question = gr.Textbox(label="Question")
        answer = gr.Textbox(label="Answer")

        ask_btn = gr.Button("Ask")
        ask_btn.click(ask_question, inputs=[caption_out, question, api_key], outputs=answer)

    return demo

demo = app()
demo.launch()
